<a href="https://colab.research.google.com/github/sriharikrishna/JDEV-Tutorial/blob/python/06MassiveInputsFewOutputs/06MassiveInputsFewOutputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistical Functions on Large Inputs

This section defines statistical quantity-of-interest functions using `jax.numpy` and applies `jax.jit` for performance on the main computation.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np  # used for initial array construction due to sequential dependency

jax.config.update("jax_enable_x64", True)

In [ ]:
def compute_avg(values):
    return jnp.mean(values)

def compute_mean_abs_deviation(values):
    avg = compute_avg(values)
    return jnp.mean(jnp.abs(values - avg))

def compute_sq_norm(values):
    return jnp.sum(jnp.square(values))

def compute_std(values):
    return jnp.std(values, ddof=0)

def abs_to_std(a, s):
    return jnp.where(s != 0, (a * a - s * s) / s, jnp.array(jnp.inf, dtype=s.dtype))

@jax.jit
def compute_qoi(values):
    average = compute_avg(values)
    mabs = compute_mean_abs_deviation(values)
    std = compute_std(values)
    a_to_s = abs_to_std(mabs, std)
    return average, mabs, std, a_to_s

In [ ]:
n = 200
values_list = [0.0] * n

values_list[n - 1] = 32.01

# Fill values in reverse order
for i in range(n - 1, 0, -1):
    current_val_i = values_list[i]
    values_list[i - 1] = 2.0 * np.sqrt(current_val_i) * i + np.exp(-current_val_i)

values = jnp.array(values_list, dtype=jnp.float64)

print(f"First 5 elements: {values[:5]}")
print(f"Last 5 elements: {values[-5:]}")

In [ ]:
average, mabs, std, a_to_s = compute_qoi(values)

print(f"Avg = {average:.6f}\t Mean abs deviation = {mabs:.6f}\t Standard deviation = {std:.6f}\t Our metric = {a_to_s:.6f}")